# Estruturação do corpus para o IRaMuTeQ — Greve USP 2023

Este notebook parte da planilha **já limpa** (com a coluna `Texto para Análise`) e gera:

1. um arquivo `.txt` estruturado no formato do IRaMuTeQ (`**** *id_X *ator_Y *grupo_Z` + texto);
2. uma nova planilha `.xlsx` com **apenas uma coluna a mais** que a original — `Texto_IRaMuTeQ` — contendo o bloco de metadados + texto usado no corpus.

**O que este notebook NÃO faz:** nenhuma nova limpeza textual. A coluna `Texto para Análise` é copiada exatamente como está para `Texto IRaMuTeQ` — sem remoção de pontuação, acentos, stopwords, sem lematização, stemming, correção ou qualquer alteração semântica.

**Como o notebook garante que nenhuma unidade se perde:**
- a planilha original (`df_original`) nunca é modificada — todo o trabalho ocorre em uma cópia (`df_iramuteq`);
- a Célula 5 conta e valida o número de linhas, IDs únicos, duplicados, textos vazios e preenchidos **antes** de qualquer arquivo ser gerado, e **interrompe a execução** (levanta erro) se encontrar IDs duplicados;
- nenhuma unidade é reordenada, removida da planilha, duplicada ou tem texto inventado;
- unidades com `Texto para Análise` vazio **permanecem na planilha de saída**, identificáveis pelo ID, mas são **excluídas apenas do `.txt`** (explicado na Célula 8) porque um bloco IRaMuTeQ sem nenhum texto após o `****` não carrega informação lexical e pode causar leitura incorreta pelo software — a lista de IDs excluídos é sempre exibida, nunca fica "escondida";
- a Célula 12 confere, ao final, se `número de unidades na planilha` = `unidades exportadas` + `unidades vazias excluídas`, e alerta caso a conta não feche.

## Célula 1 — Importação das bibliotecas

In [ ]:
import re
import os
import io
import unicodedata
from collections import Counter

import pandas as pd
import openpyxl
from google.colab import files

print("Bibliotecas carregadas com sucesso.")

## Célula 2 — Upload da planilha .xlsx (já limpa)
O arquivo original nunca é sobrescrito — todo o processamento ocorre em memória e gera arquivos novos.

In [ ]:
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Por favor, envie exatamente UM arquivo .xlsx.")

ORIGINAL_FILENAME = list(uploaded.keys())[0]
if not ORIGINAL_FILENAME.lower().endswith(('.xlsx', '.xlsm')):
    raise ValueError(f"O arquivo '{ORIGINAL_FILENAME}' não parece ser um .xlsx.")

FILE_BYTES = uploaded[ORIGINAL_FILENAME]
print(f"Arquivo carregado: {ORIGINAL_FILENAME}")
print(f"Tamanho: {len(FILE_BYTES)} bytes")

## Célula 3 — Identificação da aba `Texto Completo` e das colunas necessárias

Procura a aba `Texto Completo` (ou similar) e confirma a presença das colunas:
`ID`, `Data`, `Ator`, `Grupo do Ator`, `Transcrição Integral da Publicação`, `Texto para Análise`.

Se a detecção automática não encontrar a aba ou alguma coluna com esses nomes exatos, edite as variáveis de override abaixo.

In [ ]:
# --- Preencha manualmente apenas se a detecção automática falhar ---
SHEET_NAME_OVERRIDE = None   # ex: "Texto Completo"
COLUMN_OVERRIDES = {
    # "ID": "ID",
    # "Data": "Data",
    # "Ator": "Ator",
    # "Grupo do Ator": "Grupo do Ator",
    # "Transcrição Integral da Publicação": "Transcrição Integral da Publicação",
    # "Texto para Análise": "Texto para Análise",
}
# ---------------------------------------------------------------------

xls = pd.ExcelFile(io.BytesIO(FILE_BYTES))
sheet_names = xls.sheet_names
print("Abas encontradas:", sheet_names)

if SHEET_NAME_OVERRIDE:
    SHEET_NAME = SHEET_NAME_OVERRIDE
else:
    candidatas = [s for s in sheet_names if 'texto completo' in s.lower()]
    if not candidatas:
        candidatas = [s for s in sheet_names if 'texto' in s.lower()]
    SHEET_NAME = candidatas[0] if candidatas else sheet_names[0]

print(f"Aba selecionada: '{SHEET_NAME}'")

df_original = pd.read_excel(io.BytesIO(FILE_BYTES), sheet_name=SHEET_NAME)
print(f"Colunas encontradas na planilha: {list(df_original.columns)}")
print(f"Número de linhas encontradas: {len(df_original)}")

REQUIRED_COLUMNS = ["ID", "Data", "Ator", "Grupo do Ator",
                     "Transcrição Integral da Publicação", "Texto para Análise"]

resolved_columns = {}
missing = []
for req in REQUIRED_COLUMNS:
    if req in COLUMN_OVERRIDES:
        name = COLUMN_OVERRIDES[req]
    elif req in df_original.columns:
        name = req
    else:
        # tenta um match aproximado (case-insensitive, ignorando acentos)
        def _norm(s):
            s = unicodedata.normalize('NFKD', str(s))
            return ''.join(c for c in s if not unicodedata.combining(c)).lower().strip()
        alvo = _norm(req)
        match = [c for c in df_original.columns if _norm(c) == alvo]
        name = match[0] if match else None
    if name is None:
        missing.append(req)
    else:
        resolved_columns[req] = name

if missing:
    raise ValueError(
        "As seguintes colunas obrigatórias não foram encontradas na planilha: "
        f"{missing}. Colunas disponíveis: {list(df_original.columns)}. "
        "Preencha COLUMN_OVERRIDES acima com o nome exato e rode esta célula novamente."
    )

ID_COL = resolved_columns["ID"]
DATA_COL = resolved_columns["Data"]
ATOR_COL = resolved_columns["Ator"]
GRUPO_COL = resolved_columns["Grupo do Ator"]
TRANSCRICAO_COL = resolved_columns["Transcrição Integral da Publicação"]
TEXTO_ANALISE_COL = resolved_columns["Texto para Análise"]

print("\nColunas resolvidas:")
for k, v in resolved_columns.items():
    print(f"  {k} -> '{v}'")

print(f"\nUnidades de análise encontradas: {len(df_original)}")

## Célula 4 — Criação de `df_iramuteq`

`df_original` permanece intocado. Toda transformação seguinte ocorre em uma cópia independente.

In [ ]:
df_iramuteq = df_original.copy(deep=True)
print(f"Cópia 'df_iramuteq' criada com {len(df_iramuteq)} linhas (df_original permanece inalterado).")

## Célula 5 — Validação da base

Confere a integridade da base **antes** de qualquer geração de arquivo. Se houver IDs duplicados, a execução é **interrompida** com um erro — o notebook não segue adiante silenciosamente.

In [ ]:
total_unidades = len(df_iramuteq)
ids_unicos = df_iramuteq[ID_COL].nunique(dropna=False)
ids_duplicados_mask = df_iramuteq[ID_COL].duplicated(keep=False)
ids_duplicados = df_iramuteq.loc[ids_duplicados_mask, ID_COL].unique().tolist()

textos_vazios_mask = df_iramuteq[TEXTO_ANALISE_COL].isna() | \
    (df_iramuteq[TEXTO_ANALISE_COL].astype(str).str.strip() == '')
n_textos_vazios = int(textos_vazios_mask.sum())
n_textos_preenchidos = total_unidades - n_textos_vazios

print(f"Unidades na base: {total_unidades}")
print(f"IDs únicos: {ids_unicos}")
print(f"Textos preenchidos: {n_textos_preenchidos}")
print(f"Textos vazios: {n_textos_vazios}")

if ids_duplicados:
    raise ValueError(
        f"ERRO: foram encontrados IDs duplicados: {ids_duplicados}. "
        "Corrija a planilha de origem antes de gerar o corpus — "
        "o notebook foi interrompido de propósito para evitar um corpus incorreto."
    )
else:
    print("\nNenhum ID duplicado encontrado. Base validada.")

if n_textos_vazios > 0:
    ids_vazios = df_iramuteq.loc[textos_vazios_mask, ID_COL].tolist()
    print(f"\nATENÇÃO: {n_textos_vazios} unidade(s) sem texto em '{TEXTO_ANALISE_COL}'. "
          f"IDs: {ids_vazios}")
    print("Essas unidades permanecerão na planilha de saída, mas serão excluídas do "
          "arquivo .txt do IRaMuTeQ (ver explicação na Célula 8). Nenhum texto será inventado.")

## Célula 6 — Criação dos metadados do IRaMuTeQ

Os metadados do IRaMuTeQ não podem conter espaços. Aplicamos uma sanitização técnica que:
- remove acentos (recomendação de compatibilidade para *nomes de variável/modalidade* do IRaMuTeQ — diferente do texto do corpus, que preserva acentos integralmente);
- troca espaços por `_`;
- remove qualquer caractere que não seja letra, número ou `_`.

Tanto **Ator** quanto **Grupo do Ator** são convertidos para minúsculas com underscore (ex: `DCE` → `dce`, `Movimento estudantil` → `movimento_estudantil`).

**Sobre a variável Data:** por padrão este notebook inclui `*data_DDMMAAAA` como metadado (dia-mês-ano, sem barras). Vale registrar: esse formato ordena incorretamente em ordenação alfabética/lexicográfica (ex: `01092023` viria antes de `15012023`, mesmo sendo posterior), o que pode atrapalhar caso você queira ordenar cronologicamente pela variável dentro do IRaMuTeQ — mas não há risco de má interpretação do valor em si, já que é só um rótulo de metadado, não um campo de data processado pelo software. Se quiser voltar para `*data_AAAAMMDD` (que ordena corretamente), ou desativar a variável, ajuste `DATE_FORMAT` / `INCLUDE_DATA_VARIABLE` abaixo.

In [ ]:
INCLUDE_DATA_VARIABLE = True   # troque para False se não quiser *data_ como metadado
DATE_FORMAT = '%d%m%Y'         # DDMMAAAA. Use '%Y%m%d' (AAAAMMDD) se preferir ordenação cronológica correta.

def sanitize_metadata_value(value, lowercase=True):
    """Transforma um valor de coluna em um token seguro para metadado do IRaMuTeQ,
    sem alterar o significado da categoria (só a forma técnica)."""
    if pd.isna(value) or str(value).strip() == '':
        return 'nao_informado'
    text = str(value).strip()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))  # remove acentos
    text = re.sub(r'\s+', '_', text)
    text = re.sub(r'[^A-Za-z0-9_]', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    if lowercase:
        text = text.lower()
    return text if text else 'nao_informado'


df_iramuteq['Ator_IRaMuTeQ'] = df_iramuteq[ATOR_COL].apply(lambda v: sanitize_metadata_value(v, lowercase=True))
df_iramuteq['Grupo_IRaMuTeQ'] = df_iramuteq[GRUPO_COL].apply(lambda v: sanitize_metadata_value(v, lowercase=True))

if INCLUDE_DATA_VARIABLE:
    datas_convertidas = pd.to_datetime(df_iramuteq[DATA_COL], dayfirst=True, errors='coerce')
    n_datas_invalidas = int(datas_convertidas.isna().sum() - df_iramuteq[DATA_COL].isna().sum())
    df_iramuteq['Data_IRaMuTeQ'] = datas_convertidas.dt.strftime(DATE_FORMAT)
    if n_datas_invalidas > 0:
        print(f"AVISO: {n_datas_invalidas} data(s) não puderam ser interpretadas e ficarão "
              f"sem a variável *data_ no bloco correspondente (o restante do bloco é gerado normalmente).")
else:
    df_iramuteq['Data_IRaMuTeQ'] = None

print("Atores encontrados ->", sorted(df_iramuteq['Ator_IRaMuTeQ'].unique().tolist()))
print("Grupos encontrados  ->", sorted(df_iramuteq['Grupo_IRaMuTeQ'].unique().tolist()))
df_iramuteq[[ID_COL, ATOR_COL, 'Ator_IRaMuTeQ', GRUPO_COL, 'Grupo_IRaMuTeQ'] +
            (['Data_IRaMuTeQ'] if INCLUDE_DATA_VARIABLE else [])].head(5)

## Célula 7 — Criação da coluna `Texto IRaMuTeQ`

Cópia **exata** do conteúdo de `Texto para Análise` — nenhuma transformação textual é aplicada aqui. Células vazias permanecem vazias (nenhum texto é inventado).

In [ ]:
df_iramuteq['Texto IRaMuTeQ'] = df_iramuteq[TEXTO_ANALISE_COL]

n_iguais = (df_iramuteq['Texto IRaMuTeQ'].fillna('__NA__') == df_iramuteq[TEXTO_ANALISE_COL].fillna('__NA__')).sum()
print(f"Verificação: {n_iguais} de {len(df_iramuteq)} valores em 'Texto IRaMuTeQ' são idênticos "
      f"a '{TEXTO_ANALISE_COL}' (deve ser igual ao total de linhas).")

## Célula 8 — Geração dos blocos IRaMuTeQ e amostra das 5 primeiras unidades

Cada unidade vira um bloco:

```
**** *id_X *ator_Y *grupo_Z [*data_DDMMAAAA]
texto da unidade
```

**Sobre unidades sem texto:** um bloco `****` sem nenhum conteúdo textual depois dele não produz nenhuma ocorrência lexical no IRaMuTeQ — na prática, o software normalmente ignora o corpo vazio ou pode gerar erro de leitura dependendo da versão, mas de qualquer forma essa unidade não contribuiria em nada para a análise de palavras. Por isso, unidades com `Texto para Análise` vazio **são excluídas apenas do arquivo .txt**, e listadas explicitamente abaixo — elas continuam presentes e identificáveis na planilha `.xlsx` de saída (Célula 11).

In [ ]:
def build_block(row):
    id_val = row[ID_COL]
    try:
        id_str = str(int(id_val))
    except (ValueError, TypeError):
        id_str = str(id_val).strip()

    metadata_parts = [f"*id_{id_str}", f"*ator_{row['Ator_IRaMuTeQ']}", f"*grupo_{row['Grupo_IRaMuTeQ']}"]
    if INCLUDE_DATA_VARIABLE and pd.notna(row.get('Data_IRaMuTeQ')):
        metadata_parts.append(f"*data_{row['Data_IRaMuTeQ']}")

    header = "**** " + " ".join(metadata_parts)
    texto = str(row['Texto IRaMuTeQ']).strip()
    return header + "\n" + texto


texto_vazio_mask = df_iramuteq['Texto IRaMuTeQ'].isna() | \
    (df_iramuteq['Texto IRaMuTeQ'].astype(str).str.strip() == '')

ids_excluidos_do_txt = df_iramuteq.loc[texto_vazio_mask, ID_COL].tolist()
df_para_exportar = df_iramuteq.loc[~texto_vazio_mask].copy()

blocos = [build_block(row) for _, row in df_para_exportar.iterrows()]

# Coluna única com o bloco completo (metadados + texto), na MESMA ordem/linhas da planilha
# original. Unidades sem texto ficam com essa coluna vazia (nenhum bloco é inventado para elas).
df_iramuteq['Texto_IRaMuTeQ'] = None
df_iramuteq.loc[df_para_exportar.index, 'Texto_IRaMuTeQ'] = blocos

print(f"Unidades que serão exportadas para o corpus .txt: {len(blocos)}")
if ids_excluidos_do_txt:
    print(f"Unidades EXCLUÍDAS do .txt por não terem texto (permanecem na planilha, coluna vazia): {ids_excluidos_do_txt}")

print("\n--- Amostra das 5 primeiras unidades no formato IRaMuTeQ ---\n")
print("\n\n".join(blocos[:5]))

## Célula 9 — Geração do arquivo `.txt` em UTF-8

Cada bloco é separado do seguinte por uma linha em branco. Nenhuma tabela, aspas ou elemento extra é incluído — apenas os blocos `****` e o texto.

In [ ]:
TXT_FILENAME = "Corpus_greve_USP_2023_IRaMuTeQ.txt"

corpus_final = "\n\n".join(blocos)

with open(TXT_FILENAME, "w", encoding="utf-8") as f:
    f.write(corpus_final)

print(f"Arquivo salvo: {TXT_FILENAME}")
print(f"Codificação: UTF-8")
print(f"Total de blocos escritos: {len(blocos)}")

# Verificação de caracteres problemáticos de codificação
with open(TXT_FILENAME, "r", encoding="utf-8") as f:
    conteudo_verificacao = f.read()

n_mojibake = conteudo_verificacao.count('\ufffd')
print(f"\nCaracteres problemáticos (�) encontrados no arquivo final: {n_mojibake}")

print("\nVerificação explícita de preservação de acentos/ç:")
for caractere in ['ç', 'ã', 'é', 'ô']:
    presente = caractere in conteudo_verificacao
    print(f"  '{caractere}' presente no corpus final? {'SIM' if presente else 'NÃO — verificar!'}")

## Célula 10 — Geração dos arquivos `.txt` por ator

Além do corpus completo, geramos um `.txt` separado para cada um dos atores estudantis (`DCE`, `Correnteza`, `Juntos`, `UJC`) e um `.txt` combinado com `Reitoria` + `Instituto` juntos.

O agrupamento é feito pela versão sanitizada de `Ator` (`Ator_IRaMuTeQ`, minúscula, sem acento/espaço), a mesma usada no metadado `*ator_` de cada bloco. Cada arquivo contém exatamente os mesmos blocos (`**** *id_... ...` + texto) que já estão no corpus completo — nenhum texto é gerado ou alterado aqui, apenas filtrado e reagrupado.

Se algum grupo não tiver nenhuma unidade correspondente, o notebook avisa em vez de gerar um arquivo vazio silenciosamente.

In [ ]:
ACTOR_FILE_GROUPS = {
    "DCE": ["dce"],
    "Correnteza": ["correnteza"],
    "Juntos": ["juntos"],
    "UJC": ["ujc"],
    "Reitoria_Instituto": ["reitoria", "instituto"],
}

actor_txt_filenames = []

for group_label, actor_tokens in ACTOR_FILE_GROUPS.items():
    df_grupo = df_para_exportar.loc[df_para_exportar['Ator_IRaMuTeQ'].isin(actor_tokens)]
    blocos_grupo = df_grupo['Texto_IRaMuTeQ'].tolist()

    if not blocos_grupo:
        print(f"ATENÇÃO: nenhuma unidade encontrada para o grupo '{group_label}' "
              f"(tokens buscados: {actor_tokens}). Nenhum arquivo foi gerado para esse grupo.")
        continue

    filename = f"Corpus_greve_USP_2023_IRaMuTeQ_{group_label}.txt"
    conteudo_grupo = "\n\n".join(blocos_grupo)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(conteudo_grupo)

    actor_txt_filenames.append(filename)
    print(f"Arquivo salvo: {filename}  |  Unidades incluídas: {len(blocos_grupo)}  |  IDs: {df_grupo[ID_COL].tolist()}")

print(f"\nTotal de arquivos por ator/grupo gerados: {len(actor_txt_filenames)}")

## Célula 11 — Geração da nova planilha `.xlsx`

A planilha de saída tem **exatamente uma coluna a mais** que a original: `Texto_IRaMuTeQ`, contendo o bloco completo (metadados `**** *id_... *ator_... *grupo_...` + texto) que foi escrito no `.txt` para cada unidade. Unidades sem texto ficam com essa coluna vazia. O arquivo original enviado no upload não é alterado — este é um arquivo novo.

In [ ]:
XLSX_FILENAME = "Banco_greve_USP_2023_IRaMuTeQ.xlsx"

colunas_saida = list(df_original.columns) + ['Texto_IRaMuTeQ']

df_iramuteq[colunas_saida].to_excel(XLSX_FILENAME, sheet_name=SHEET_NAME[:31], index=False)

print(f"Planilha salva: {XLSX_FILENAME}")
print(f"Colunas incluídas ({len(colunas_saida)} no total, {len(df_original.columns)} originais + 1 nova): {colunas_saida}")

## Célula 12 — Relatório final de validação

In [ ]:
n_blocos_gerados = len(blocos)
n_exportadas = len(df_para_exportar)
soma_confere = (n_exportadas + len(ids_excluidos_do_txt)) == total_unidades

print("="*70)
print("RELATÓRIO FINAL DE VALIDAÇÃO")
print("="*70)
print(f"Arquivo original utilizado:              {ORIGINAL_FILENAME}")
print(f"Aba utilizada:                            {SHEET_NAME}")
print(f"Número total de unidades na base:         {total_unidades}")
print(f"Número de IDs únicos:                     {ids_unicos}")
print(f"Número de IDs duplicados:                 {len(ids_duplicados)}")
print(f"Número de textos preenchidos:              {n_textos_preenchidos}")
print(f"Número de textos vazios:                  {n_textos_vazios}")
print(f"Número de unidades exportadas para o .txt: {n_exportadas}")
print(f"Atores encontrados:                       {sorted(df_iramuteq['Ator_IRaMuTeQ'].unique().tolist())}")
print(f"Grupos encontrados:                       {sorted(df_iramuteq['Grupo_IRaMuTeQ'].unique().tolist())}")
print(f"Número de blocos gerados no .txt completo: {n_blocos_gerados}")
print(f"Arquivos por ator/grupo gerados:            {actor_txt_filenames}")
print(f"Arquivo .txt completo salvo em UTF-8:       Sim ({TXT_FILENAME})")
print(f"Caracteres 'ç' e acentos preservados:       Sim (ver verificação da Célula 9)")
print("-"*70)
print(f"Confere: unidades totais ({total_unidades}) == exportadas ({n_exportadas}) "
      f"+ excluídas por texto vazio ({len(ids_excluidos_do_txt)})?  "
      f"{'SIM' if soma_confere else 'NÃO — INVESTIGAR!'}")
if not soma_confere:
    print("ALERTA: a contagem não fechou. Não prossiga sem investigar a causa.")
print("="*70)

## Célula 13 — Download dos arquivos

Ao importar qualquer um dos `.txt` no IRaMuTeQ, selecione a codificação **UTF-8 / UTF-8 all languages**. Não altere para ASCII, ANSI ou outra opção, sob risco de corromper os acentos do português.

In [ ]:
files.download(TXT_FILENAME)
for fname in actor_txt_filenames:
    files.download(fname)
files.download(XLSX_FILENAME)